<a href="https://colab.research.google.com/github/Ratludu/Backpack-Prediction-Challenge/blob/main/Backpack_Prices_LB_38.84599.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import locale
def getpreferredencoding(do_setlocale = True):
    return "UTF-8"
locale.getpreferredencoding = getpreferredencoding

In [2]:
from google.colab import userdata
import os
os.environ['KAGGLE_USERNAME'] = userdata.get('kaggleusername')
os.environ['KAGGLE_KEY'] = userdata.get('kaggleapi')

competition = 'playground-series-s5e2'

!kaggle competitions download -c {competition}

!unzip "{competition}.zip"

 96% 89.0M/92.7M [00:00<00:00, 174MB/s]
100% 92.7M/92.7M [00:00<00:00, 140MB/s]
Archive:  playground-series-s5e2.zip
  inflating: sample_submission.csv   
  inflating: test.csv                
  inflating: train.csv               
  inflating: training_extra.csv      


In [3]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("souradippal/student-bag-price-prediction-dataset")

print("Path to dataset files:", path)

100%|██████████| 1.23M/1.23M [00:00<00:00, 106MB/s]

Extracting files...
Path to dataset files: /root/.cache/kagglehub/datasets/souradippal/student-bag-price-prediction-dataset/versions/1


In [4]:
!pip install dask-cuda==24.12.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.4/134.4 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 46.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 244.5/244.5 kB 26.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 69.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.0/47.0 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 kB 3.9 MB/s eta 0:00:00
  Attempting uninstall: dask
    Found existing installation: dask 2024.10.0
    Uninstalling dask-2024.10.0:
      Successfully uninstalled dask-2024.10.0


In [8]:
!git clone https://github.com/rapidsai/rapidsai-csp-utils.git
!python rapidsai-csp-utils/colab/pip-install.py

fatal: destination path 'rapidsai-csp-utils' already exists and is not an empty directory.
Installing RAPIDS remaining 24.12.* libraries
Using Python 3.11.11 environment at: /usr
Resolved 154 packages in 575ms
 Downloaded ucx-py-cu12
 Downloaded cuspatial-cu12
 Downloaded libucx-cu12
 Downloaded datashader
 Downloaded libcuspatial-cu12
 Downloaded cucim-cu12
 Downloaded scikit-image
 Downloaded raft-dask-cu12
 Downloaded cuvs-cu12
 Downloaded cuml-cu12
 Downloaded cugraph-cu12
Prepared 18 packages in 26.76s
Uninstalled 1 package in 21ms
Installed 21 packages in 21ms
 + cucim-cu12==24.12.0
 + cugraph-cu12==24.12.0
 + cuml-cu12==24.12.0
 + cuproj-cu12==24.12.0
 + cuspatial-cu12==24.12.0
 + cuvs-cu12==24.12.0
 + cuxfilter-cu12==24.12.0
 + dask-cudf-cu12==24.12.0
 + datashader==0.17.0
 + distributed-ucxx-cu12==0.41.0
 + jupyter-server-proxy==4.4.0
 + libcuspatial-cu12==24.12.0
 + libucx-cu12==1.17.0.post1
 + libucxx-cu12==0.41.0
 + pyct==0.5.0
 + raft-dask-cu12==24.12.0
 - scikit-image==0.

In [6]:
!pip install catboost
!pip install optuna
!pip install scikit-learn
!pip install numpy
!pip install seaborn
!pip install matplotlib
!pip install pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.7/98.7 MB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.6/383.6 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 233.6/233.6 kB 25.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.5/78.5 kB 10.5 MB/s eta 0:00:00


In [9]:
import pandas as pd
import numpy as np
from numpy import random
from cuml.preprocessing import TargetEncoder
from catboost import CatBoostRegressor
from sklearn.model_selection import cross_validate, cross_val_predict
from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold
from sklearn.linear_model import Ridge, Lasso, LinearRegression, ElasticNet
import matplotlib.pyplot as plt
import seaborn as sns
import optuna
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
import warnings
warnings.filterwarnings('ignore')

In [10]:
class config:
    # data links
    train_link = "train.csv"
    train_ex_link = "training_extra.csv"
    test_link = "test.csv"
    sub_link = "sample_submission.csv"

    # create folds config

    n_splits = 25

    # Ignore Columns

    col_ignore = ["id", "Price"]
    num_cols = ["Weight Capacity (kg)"]
    # target

    submit = True

    target = "Price"

    add_original = False

In [11]:
def rmse(y_true, y_pred):
    error = 0

    for yt, yp in zip(y_true, y_pred):
        error += (yt - yp) ** 2

    m = np.sqrt(error / len(y_true))

    return m

In [12]:
def random_columns(columns):

  # Generate random number for how many columns we want to concat
  rand_num = np.random.randint(2,5)

  # choose the columns from the list of columns with no repeats
  rand_cols = []
  for i in range(rand_num):
    col = np.random.choice(columns)
    while col in rand_cols:
      col = np.random.choice(columns)
    rand_cols.append(col)

  # return a list of the columns

  return "-".join(col for col in rand_cols),rand_cols


In [13]:
train = pd.read_csv(config.train_link)
train_ex = pd.read_csv(config.train_ex_link)
test = pd.read_csv(config.test_link)
original = pd.read_csv(path+"/Noisy_Student_Bag_Price_Prediction_Dataset.csv")
original.dropna(subset=['Price'],inplace = True)

In [14]:
train = pd.concat([train,train_ex, original], axis = 0, ignore_index = True)

In [44]:
# train = train.sample(frac = 0.3, random_state = 42, ignore_index = True)

In [15]:
kf = KFold(n_splits = config.n_splits, shuffle = True, random_state = 42)

drop = ['Brand', 'Material', 'Size', 'Compartments', 'Laptop Compartment', 'Color', 'Waterproof']
added_fe = ['size-laptop compartment_fe','Color-waterproof_fe','weightcapacity-color_fe']
oof = np.zeros(len(train))
preds = np.zeros(len(test))
features = [col for col in train.columns if col not in config.col_ignore]
cats = [col for col in features if col not in config.num_cols]
cats.extend(added_fe)
m = []
for fold, (train_idx, test_idx) in enumerate(kf.split(train)):

    x_train, x_val = train.loc[train_idx, features].copy(), train.loc[test_idx, features].copy()
    y_train, y_val = train.loc[train_idx, config.target].copy(), train.loc[test_idx, config.target].copy()
    x_test = test[features].copy()

    TE = TargetEncoder(n_folds=27, smooth=21, split_method = 'random', stat = 'mean')


    # adding extra features

    x_train["size-laptop compartment_fe"] = x_train['Size'].astype('str')+x_train['Laptop Compartment'].astype('str')
    x_val["size-laptop compartment_fe"] = x_val['Size'].astype('str')+x_val['Laptop Compartment'].astype('str')
    x_test["size-laptop compartment_fe"] = x_test['Size'].astype('str')+x_test['Laptop Compartment'].astype('str')

    x_train["Color-waterproof_fe"] = x_train['Color'].astype('str')+x_train['Waterproof'].astype('str')
    x_val["Color-waterproof_fe"] = x_val['Color'].astype('str')+x_val['Waterproof'].astype('str')
    x_test["Color-waterproof_fe"] = x_test['Color'].astype('str')+x_test['Waterproof'].astype('str')

    x_train["weightcapacity-color_fe"] = x_train['Weight Capacity (kg)'].astype('str') + x_train['Color'].astype('str')
    x_val["weightcapacity-color_fe"] = x_val['Weight Capacity (kg)'].astype('str') + x_val['Color'].astype('str')
    x_test["weightcapacity-color_fe"] = x_test['Weight Capacity (kg)'].astype('str')+x_test['Color'].astype('str')

    x_train["weight_log"] = np.log1p(x_train["Weight Capacity (kg)"])**2
    x_val["weight_log"] = np.log1p(x_val["Weight Capacity (kg)"])**2
    x_test["weight_log"] = np.log1p(x_test["Weight Capacity (kg)"])**2

    for col in added_fe:
      TE.fit(x_train[col], y_train)
      x_train[f'{col}_TE'] = TE.transform(x_train[col])
      x_val[f'{col}_TE'] = TE.transform(x_val[col])
      x_test[f'{col}_TE'] = TE.transform(x_test[col])

    for col in features:
        TE.fit(x_train[col], y_train)
        x_train[f'{col}_TE'] = TE.transform(x_train[col])
        x_val[f'{col}_TE'] = TE.transform(x_val[col])
        x_test[f'{col}_TE'] = TE.transform(x_test[col])

    x_train["colorxweight"] = x_train['Color_TE']*x_train['Weight Capacity (kg)_TE']
    x_val["colorxweight"] = x_val['Color_TE']*x_val['Weight Capacity (kg)_TE']
    x_test["colorxweight"] = x_test['Color_TE']*x_test['Weight Capacity (kg)_TE']

    x_train["Sizexweight"] = x_train['Size_TE']*x_train['Weight Capacity (kg)_TE']
    x_val["Sizexweight"] = x_val['Size_TE']*x_val['Weight Capacity (kg)_TE']
    x_test["Sizexweight"] = x_test['Size_TE']*x_test['Weight Capacity (kg)_TE']

    x_train["Sizexbrand"] = x_train['Size_TE']*x_train['Brand_TE']
    x_val["Sizexbrand"] = x_val['Size_TE']*x_val['Brand_TE']
    x_test["Sizexbrand"] = x_test['Size_TE']*x_test['Brand_TE']

    x_train["Materialxweight"] = x_train['Material_TE']*x_train['Weight Capacity (kg)_TE']
    x_val["Materialxweight"] = x_val['Material_TE']*x_val['Weight Capacity (kg)_TE']
    x_test["Materialxweight"] = x_test['Material_TE']*x_test['Weight Capacity (kg)_TE']

    x_train['Material_TE-Color_TE'] = x_train['Material_TE']*x_train['Color_TE']
    x_val['Material_TE-Color_TE'] = x_val['Material_TE']*x_val['Color_TE']
    x_test['Material_TE-Color_TE'] = x_test['Material_TE']*x_test['Color_TE']

    x_train['Sizexbrand-Materialxweight'] = (x_train['Sizexbrand']-x_train['Materialxweight'])**2
    x_val['Sizexbrand-Materialxweight'] = (x_val['Sizexbrand']-x_val['Materialxweight'])**2
    x_test['Sizexbrand-Materialxweight'] = (x_test['Sizexbrand']-x_test['Materialxweight'])**2

    x_train['Sizexweight-Laptop Compartment_TE'] = x_train['Sizexweight']/x_train['Laptop Compartment_TE']
    x_val['Sizexweight-Laptop Compartment_TE'] = x_val['Sizexweight']/x_val['Laptop Compartment_TE']
    x_test['Sizexweight-Laptop Compartment_TE'] = x_test['Sizexweight']/x_test['Laptop Compartment_TE']

    x_train['Color_TE-Waterproof_TE'] = x_train['Color_TE']**2+x_train['Waterproof_TE']*x_train['Color_TE']
    x_val['Color_TE-Waterproof_TE'] = x_val['Color_TE']**2+x_val['Waterproof_TE']*x_val['Color_TE']
    x_test['Color_TE-Waterproof_TE'] = x_test['Color_TE']**2+x_test['Waterproof_TE']*x_test['Color_TE']

    for cat in cats:
        x_train[cat] =  x_train[cat].fillna("MISSING")
        x_val[cat] = x_val[cat].fillna("MISSING")
        x_test[cat] = x_test[cat].fillna("MISSING")
        x_train[cat] =  x_train[cat].astype('str')
        x_val[cat] = x_val[cat].astype('str')
        x_test[cat] = x_test[cat].astype('str')

    print(x_train.columns)

    model = CatBoostRegressor(
                              learning_rate = 0.11509572776170199,
                              #l2_leaf_reg=5,
                              iterations = 2000,
                              task_type = "GPU",
                              grow_policy = 'Lossguide',
                              random_state = 42,
                              cat_features = cats,
                              #early_stopping_rounds = 25,
                              verbose = 250,
                              loss_function='RMSE')

    model.fit(x_train, y_train, eval_set=(x_val,y_val))

    val_preds = model.predict(x_val)

    oof[test_idx] = val_preds

    preds += model.predict(x_test)/config.n_splits

    score = rmse(y_val, val_preds)

    m.append(score)

    print(f'Fold: {fold+1}, Score: {score}')

print(f"The average CV is {np.average(m)}")

Index(['Brand', 'Material', 'Size', 'Compartments', 'Laptop Compartment',
       'Waterproof', 'Style', 'Color', 'Weight Capacity (kg)',
       'size-laptop compartment_fe', 'Color-waterproof_fe',
       'weightcapacity-color_fe', 'weight_log',
       'size-laptop compartment_fe_TE', 'Color-waterproof_fe_TE',
       'weightcapacity-color_fe_TE', 'Brand_TE', 'Material_TE', 'Size_TE',
       'Compartments_TE', 'Laptop Compartment_TE', 'Waterproof_TE', 'Style_TE',
       'Color_TE', 'Weight Capacity (kg)_TE', 'colorxweight', 'Sizexweight',
       'Sizexbrand', 'Materialxweight', 'Material_TE-Color_TE',
       'Sizexbrand-Materialxweight', 'Sizexweight-Laptop Compartment_TE',
       'Color_TE-Waterproof_TE'],
      dtype='object')
0:	learn: 38.8872737	test: 38.8861687	best: 38.8861687 (0)	total: 45.9ms	remaining: 1m 31s
250:	learn: 38.6213504	test: 38.6496502	best: 38.6492382 (197)	total: 7.96s	remaining: 55.5s
500:	learn: 38.5936191	test: 38.6475133	best: 38.6474583 (498)	total: 15.2s	rem

In [19]:
kf = KFold(n_splits = config.n_splits, shuffle = True, random_state = 42)

drop = ['Brand', 'Material', 'Size', 'Compartments', 'Laptop Compartment', 'Color', 'Waterproof']
added_fe = ['size-laptop compartment_fe','Color-waterproof_fe','weightcapacity-color_fe']
cat_oof = np.zeros(len(train))
cat_preds = np.zeros(len(test))
features = [col for col in train.columns if col not in config.col_ignore]
cats = [col for col in features if col not in config.num_cols]
cats.extend(added_fe)
m = []
for fold, (train_idx, test_idx) in enumerate(kf.split(train)):

    x_train, x_val = train.loc[train_idx, features].copy(), train.loc[test_idx, features].copy()
    y_train, y_val = train.loc[train_idx, config.target].copy(), train.loc[test_idx, config.target].copy()
    x_test = test[features].copy()

    TE = TargetEncoder(n_folds=27, smooth=21, split_method = 'random', stat = 'mean')


    # adding extra features

    x_train["size-laptop compartment_fe"] = x_train['Size'].astype('str')+x_train['Laptop Compartment'].astype('str')
    x_val["size-laptop compartment_fe"] = x_val['Size'].astype('str')+x_val['Laptop Compartment'].astype('str')
    x_test["size-laptop compartment_fe"] = x_test['Size'].astype('str')+x_test['Laptop Compartment'].astype('str')

    x_train["Color-waterproof_fe"] = x_train['Color'].astype('str')+x_train['Waterproof'].astype('str')
    x_val["Color-waterproof_fe"] = x_val['Color'].astype('str')+x_val['Waterproof'].astype('str')
    x_test["Color-waterproof_fe"] = x_test['Color'].astype('str')+x_test['Waterproof'].astype('str')

    x_train["weightcapacity-color_fe"] = x_train['Weight Capacity (kg)'].astype('str') + x_train['Color'].astype('str')
    x_val["weightcapacity-color_fe"] = x_val['Weight Capacity (kg)'].astype('str') + x_val['Color'].astype('str')
    x_test["weightcapacity-color_fe"] = x_test['Weight Capacity (kg)'].astype('str')+x_test['Color'].astype('str')

    x_train["weight_log"] = np.log1p(x_train["Weight Capacity (kg)"])**2
    x_val["weight_log"] = np.log1p(x_val["Weight Capacity (kg)"])**2
    x_test["weight_log"] = np.log1p(x_test["Weight Capacity (kg)"])**2

    for col in added_fe:
      TE.fit(x_train[col], y_train)
      x_train[f'{col}_TE'] = TE.transform(x_train[col])
      x_val[f'{col}_TE'] = TE.transform(x_val[col])
      x_test[f'{col}_TE'] = TE.transform(x_test[col])

    for col in features:
        TE.fit(x_train[col], y_train)
        x_train[f'{col}_TE'] = TE.transform(x_train[col])
        x_val[f'{col}_TE'] = TE.transform(x_val[col])
        x_test[f'{col}_TE'] = TE.transform(x_test[col])

    x_train["colorxweight"] = x_train['Color_TE']*x_train['Weight Capacity (kg)_TE']
    x_val["colorxweight"] = x_val['Color_TE']*x_val['Weight Capacity (kg)_TE']
    x_test["colorxweight"] = x_test['Color_TE']*x_test['Weight Capacity (kg)_TE']

    x_train["Sizexweight"] = x_train['Size_TE']*x_train['Weight Capacity (kg)_TE']
    x_val["Sizexweight"] = x_val['Size_TE']*x_val['Weight Capacity (kg)_TE']
    x_test["Sizexweight"] = x_test['Size_TE']*x_test['Weight Capacity (kg)_TE']

    x_train["Sizexbrand"] = x_train['Size_TE']*x_train['Brand_TE']
    x_val["Sizexbrand"] = x_val['Size_TE']*x_val['Brand_TE']
    x_test["Sizexbrand"] = x_test['Size_TE']*x_test['Brand_TE']

    x_train["Materialxweight"] = x_train['Material_TE']*x_train['Weight Capacity (kg)_TE']
    x_val["Materialxweight"] = x_val['Material_TE']*x_val['Weight Capacity (kg)_TE']
    x_test["Materialxweight"] = x_test['Material_TE']*x_test['Weight Capacity (kg)_TE']

    x_train['Material_TE-Color_TE'] = x_train['Material_TE']*x_train['Color_TE']
    x_val['Material_TE-Color_TE'] = x_val['Material_TE']*x_val['Color_TE']
    x_test['Material_TE-Color_TE'] = x_test['Material_TE']*x_test['Color_TE']

    x_train['Sizexbrand-Materialxweight'] = (x_train['Sizexbrand']-x_train['Materialxweight'])**2
    x_val['Sizexbrand-Materialxweight'] = (x_val['Sizexbrand']-x_val['Materialxweight'])**2
    x_test['Sizexbrand-Materialxweight'] = (x_test['Sizexbrand']-x_test['Materialxweight'])**2

    x_train['Sizexweight-Laptop Compartment_TE'] = x_train['Sizexweight']/x_train['Laptop Compartment_TE']
    x_val['Sizexweight-Laptop Compartment_TE'] = x_val['Sizexweight']/x_val['Laptop Compartment_TE']
    x_test['Sizexweight-Laptop Compartment_TE'] = x_test['Sizexweight']/x_test['Laptop Compartment_TE']

    x_train['Color_TE-Waterproof_TE'] = x_train['Color_TE']**2+x_train['Waterproof_TE']*x_train['Color_TE']
    x_val['Color_TE-Waterproof_TE'] = x_val['Color_TE']**2+x_val['Waterproof_TE']*x_val['Color_TE']
    x_test['Color_TE-Waterproof_TE'] = x_test['Color_TE']**2+x_test['Waterproof_TE']*x_test['Color_TE']

    for cat in cats:
        x_train[cat] =  x_train[cat].fillna("MISSING")
        x_val[cat] = x_val[cat].fillna("MISSING")
        x_test[cat] = x_test[cat].fillna("MISSING")
        x_train[cat] =  x_train[cat].astype('str')
        x_val[cat] = x_val[cat].astype('str')
        x_test[cat] = x_test[cat].astype('str')

    print(x_train.columns)

    model = CatBoostRegressor(
                              learning_rate = 0.11509572776170199,
                              #l2_leaf_reg=5,
                              per_float_feature_quantization='8:border_count=1024',
                              iterations = 2000,
                              task_type = "GPU",
                              grow_policy = 'Lossguide',
                              random_state = 42,
                              cat_features = cats,
                              #early_stopping_rounds = 25,
                              verbose = 250,
                              loss_function='RMSE')

    model.fit(x_train, y_train, eval_set=(x_val,y_val))

    val_preds = model.predict(x_val)

    cat_oof[test_idx] = val_preds

    cat_preds += model.predict(x_test)/config.n_splits

    score = rmse(y_val, val_preds)

    m.append(score)

    print(f'Fold: {fold+1}, Score: {score}')

print(f"The average CV is {np.average(m)}")

Index(['Brand', 'Material', 'Size', 'Compartments', 'Laptop Compartment',
       'Waterproof', 'Style', 'Color', 'Weight Capacity (kg)',
       'size-laptop compartment_fe', 'Color-waterproof_fe',
       'weightcapacity-color_fe', 'weight_log',
       'size-laptop compartment_fe_TE', 'Color-waterproof_fe_TE',
       'weightcapacity-color_fe_TE', 'Brand_TE', 'Material_TE', 'Size_TE',
       'Compartments_TE', 'Laptop Compartment_TE', 'Waterproof_TE', 'Style_TE',
       'Color_TE', 'Weight Capacity (kg)_TE', 'colorxweight', 'Sizexweight',
       'Sizexbrand', 'Materialxweight', 'Material_TE-Color_TE',
       'Sizexbrand-Materialxweight', 'Sizexweight-Laptop Compartment_TE',
       'Color_TE-Waterproof_TE'],
      dtype='object')
0:	learn: 38.8872313	test: 38.8861535	best: 38.8861535 (0)	total: 41.5ms	remaining: 1m 22s
250:	learn: 38.6178571	test: 38.6457410	best: 38.6456796 (248)	total: 8.29s	remaining: 57.8s
500:	learn: 38.5883360	test: 38.6444729	best: 38.6443091 (451)	total: 15.7s	rem

In [48]:
kf = KFold(n_splits = config.n_splits, shuffle = True, random_state = 42)

drop = ['Brand', 'Material', 'Size', 'Compartments', 'Laptop Compartment', 'Color', 'Waterproof']
added_fe = ['size-laptop compartment_fe','Color-waterproof_fe','weightcapacity-color_fe']
lgbm_oof = np.zeros(len(train))
lgbm_preds = np.zeros(len(test))
features = [col for col in train.columns if col not in config.col_ignore]
cats = [col for col in features if col not in config.num_cols]
cats.extend(added_fe)
m = []
for fold, (train_idx, test_idx) in enumerate(kf.split(train)):

    x_train, x_val = train.loc[train_idx, features].copy(), train.loc[test_idx, features].copy()
    y_train, y_val = train.loc[train_idx, config.target].copy(), train.loc[test_idx, config.target].copy()
    x_test = test[features].copy()

    TE = TargetEncoder(n_folds=27, smooth=21, split_method = 'random', stat = 'mean')


    # adding extra features

    x_train["size-laptop compartment_fe"] = x_train['Size'].astype('str')+x_train['Laptop Compartment'].astype('str')
    x_val["size-laptop compartment_fe"] = x_val['Size'].astype('str')+x_val['Laptop Compartment'].astype('str')
    x_test["size-laptop compartment_fe"] = x_test['Size'].astype('str')+x_test['Laptop Compartment'].astype('str')

    x_train["Color-waterproof_fe"] = x_train['Color'].astype('str')+x_train['Waterproof'].astype('str')
    x_val["Color-waterproof_fe"] = x_val['Color'].astype('str')+x_val['Waterproof'].astype('str')
    x_test["Color-waterproof_fe"] = x_test['Color'].astype('str')+x_test['Waterproof'].astype('str')

    x_train["weightcapacity-color_fe"] = x_train['Weight Capacity (kg)'].astype('str') + x_train['Color'].astype('str')
    x_val["weightcapacity-color_fe"] = x_val['Weight Capacity (kg)'].astype('str') + x_val['Color'].astype('str')
    x_test["weightcapacity-color_fe"] = x_test['Weight Capacity (kg)'].astype('str')+x_test['Color'].astype('str')

    x_train["weight_log"] = np.log1p(x_train["Weight Capacity (kg)"])**2
    x_val["weight_log"] = np.log1p(x_val["Weight Capacity (kg)"])**2
    x_test["weight_log"] = np.log1p(x_test["Weight Capacity (kg)"])**2

    for col in added_fe:
      TE.fit(x_train[col], y_train)
      x_train[f'{col}_TE'] = TE.transform(x_train[col])
      x_val[f'{col}_TE'] = TE.transform(x_val[col])
      x_test[f'{col}_TE'] = TE.transform(x_test[col])

    for col in features:
        TE.fit(x_train[col], y_train)
        x_train[f'{col}_TE'] = TE.transform(x_train[col])
        x_val[f'{col}_TE'] = TE.transform(x_val[col])
        x_test[f'{col}_TE'] = TE.transform(x_test[col])

    x_train["colorxweight"] = x_train['Color_TE']*x_train['Weight Capacity (kg)_TE']
    x_val["colorxweight"] = x_val['Color_TE']*x_val['Weight Capacity (kg)_TE']
    x_test["colorxweight"] = x_test['Color_TE']*x_test['Weight Capacity (kg)_TE']

    x_train["Sizexweight"] = x_train['Size_TE']*x_train['Weight Capacity (kg)_TE']
    x_val["Sizexweight"] = x_val['Size_TE']*x_val['Weight Capacity (kg)_TE']
    x_test["Sizexweight"] = x_test['Size_TE']*x_test['Weight Capacity (kg)_TE']

    x_train["Sizexbrand"] = x_train['Size_TE']*x_train['Brand_TE']
    x_val["Sizexbrand"] = x_val['Size_TE']*x_val['Brand_TE']
    x_test["Sizexbrand"] = x_test['Size_TE']*x_test['Brand_TE']

    x_train["Materialxweight"] = x_train['Material_TE']*x_train['Weight Capacity (kg)_TE']
    x_val["Materialxweight"] = x_val['Material_TE']*x_val['Weight Capacity (kg)_TE']
    x_test["Materialxweight"] = x_test['Material_TE']*x_test['Weight Capacity (kg)_TE']

    x_train['Material_TE-Color_TE'] = x_train['Material_TE']*x_train['Color_TE']
    x_val['Material_TE-Color_TE'] = x_val['Material_TE']*x_val['Color_TE']
    x_test['Material_TE-Color_TE'] = x_test['Material_TE']*x_test['Color_TE']

    x_train['Sizexbrand-Materialxweight'] = (x_train['Sizexbrand']-x_train['Materialxweight'])**2
    x_val['Sizexbrand-Materialxweight'] = (x_val['Sizexbrand']-x_val['Materialxweight'])**2
    x_test['Sizexbrand-Materialxweight'] = (x_test['Sizexbrand']-x_test['Materialxweight'])**2

    x_train['Sizexweight-Laptop Compartment_TE'] = x_train['Sizexweight']/x_train['Laptop Compartment_TE']
    x_val['Sizexweight-Laptop Compartment_TE'] = x_val['Sizexweight']/x_val['Laptop Compartment_TE']
    x_test['Sizexweight-Laptop Compartment_TE'] = x_test['Sizexweight']/x_test['Laptop Compartment_TE']

    x_train['Color_TE-Waterproof_TE'] = x_train['Color_TE']**2+x_train['Waterproof_TE']*x_train['Color_TE']
    x_val['Color_TE-Waterproof_TE'] = x_val['Color_TE']**2+x_val['Waterproof_TE']*x_val['Color_TE']
    x_test['Color_TE-Waterproof_TE'] = x_test['Color_TE']**2+x_test['Waterproof_TE']*x_test['Color_TE']

    for cat in cats:
        x_train[cat] =  x_train[cat].fillna("MISSING")
        x_val[cat] = x_val[cat].fillna("MISSING")
        x_test[cat] = x_test[cat].fillna("MISSING")
        x_train[cat] =  x_train[cat].astype('category')
        x_val[cat] = x_val[cat].astype('category')
        x_test[cat] = x_test[cat].astype('category')

    print(x_train.columns)

    model = LGBMRegressor(verbose=-1)

    model.fit(x_train, y_train, eval_set=[(x_val, y_val)])

    val_preds = model.predict(x_val)

    lgbm_oof[test_idx] = val_preds

    lgbm_preds += model.predict(x_test)/config.n_splits

    score = rmse(y_val, val_preds)

    m.append(score)

    print(f'Fold: {fold+1}, Score: {score}')

print(f"The average CV is {np.average(m)}")

Index(['Brand', 'Material', 'Size', 'Compartments', 'Laptop Compartment',
       'Waterproof', 'Style', 'Color', 'Weight Capacity (kg)',
       'size-laptop compartment_fe', 'Color-waterproof_fe',
       'weightcapacity-color_fe', 'weight_log',
       'size-laptop compartment_fe_TE', 'Color-waterproof_fe_TE',
       'weightcapacity-color_fe_TE', 'Brand_TE', 'Material_TE', 'Size_TE',
       'Compartments_TE', 'Laptop Compartment_TE', 'Waterproof_TE', 'Style_TE',
       'Color_TE', 'Weight Capacity (kg)_TE', 'colorxweight', 'Sizexweight',
       'Sizexbrand', 'Materialxweight', 'Material_TE-Color_TE',
       'Sizexbrand-Materialxweight', 'Sizexweight-Laptop Compartment_TE',
       'Color_TE-Waterproof_TE'],
      dtype='object')
Fold: 1, Score: 38.683708130628084
Index(['Brand', 'Material', 'Size', 'Compartments', 'Laptop Compartment',
       'Waterproof', 'Style', 'Color', 'Weight Capacity (kg)',
       'size-laptop compartment_fe', 'Color-waterproof_fe',
       'weightcapacity-color_fe

In [23]:
# Ensemble with ridge
train_meta = pd.DataFrame()
train_meta['CatBoost'] = oof
train_meta['CatBoost2'] = cat_oof
train_meta[config.target] = train[config.target]

test_meta = pd.DataFrame()
test_meta['CatBoost'] = preds
test_meta['CatBoost2'] = cat_preds

In [24]:
kf = KFold(n_splits = config.n_splits, shuffle = True, random_state = 42)


final_preds = np.zeros(len(test))
features = ['CatBoost', 'CatBoost2']
m = []
for fold, (train_idx, test_idx) in enumerate(kf.split(train_meta)):

    x_train, x_val = train_meta.loc[train_idx, features].copy(), train_meta.loc[test_idx, features].copy()
    y_train, y_val = train_meta.loc[train_idx, config.target].copy(), train_meta.loc[test_idx, config.target].copy()
    x_test = test_meta[features].copy()

    model = Ridge()

    model.fit(x_train, y_train)

    val_preds = model.predict(x_val)

    final_preds += model.predict(x_test)/config.n_splits

    score = rmse(y_val, val_preds)
    m.append(score)

    print(f'Fold: {fold+1}, Score: {score}')

print(f"The average CV is {np.average(m)}")

Fold: 1, Score: 38.644407624902165
Fold: 2, Score: 38.62895835923888
Fold: 3, Score: 38.62081934327169
Fold: 4, Score: 38.59953321876073
Fold: 5, Score: 38.71667814225177
Fold: 6, Score: 38.58734560747046
Fold: 7, Score: 38.57052690002137
Fold: 8, Score: 38.668039172960164
Fold: 9, Score: 38.65408800853942
Fold: 10, Score: 38.657541361707274
Fold: 11, Score: 38.68230804480086
Fold: 12, Score: 38.706113168225855
Fold: 13, Score: 38.61799683972727
Fold: 14, Score: 38.718442447407675
Fold: 15, Score: 38.74421120172001
Fold: 16, Score: 38.64196174346047
Fold: 17, Score: 38.60302712790454
Fold: 18, Score: 38.58185768569726
Fold: 19, Score: 38.5767509051066
Fold: 20, Score: 38.622237573040216
Fold: 21, Score: 38.61123761695932
Fold: 22, Score: 38.70070691131668
Fold: 23, Score: 38.68176316684825
Fold: 24, Score: 38.66903247614985
Fold: 25, Score: 38.74382956471554
The average CV is 38.64997656848817


In [ ]:
from google.colab import runtime
runtime.unassign()

In [25]:
submission = pd.read_csv(config.sub_link)
submission[config.target] = final_preds
submission.to_csv("submission.csv", index = False)

submission

,id,Price
0,300000,81.920922
1,300001,82.600374
2,300002,88.795677
3,300003,80.207527
4,300004,79.338181
...,...,...
199995,499995,81.541390
199996,499996,77.554726
199997,499997,82.689694
199998,499998,82.229149


In [26]:
if config.submit:
  !kaggle competitions submit -c {competition} -f submission.csv -m 'Submission with cv 38.64997656848817'

100% 4.74M/4.74M [00:01<00:00, 4.33MB/s]
Successfully submitted to Backpack Prediction Challenge

In [ ]:
!kaggle competitions submissions -c {competition}